# Soup Kernel candidate: a fixed-commit T4 run

Set Kaggle's accelerator to **GPU T4 x2** and turn Internet on. The task uses only GPU 0; leave both GPUs idle apart from these commands. This scaffold uses the official notebook's package installation and the unchanged official harness.

Fill in the **platform-created team repository** and its **full frozen 40-character commit** below. Then Run All. A new checkout is created for each run. Failed setup or CUDA tests stop execution before training. Active source settings are tested by default; `INCLUDE_EXPERIMENTS=True` also checks disabled PTX/storage experiments.

For a private repository, optionally add a read-only `GITHUB_TOKEN` Kaggle secret. An optional `HF_TOKEN` secret can avoid model-download rate limits. Neither token is printed or written to the repository. This notebook does not register accounts or buy compute.

For submission, save a public **Save & Run All** version at the final commit and put that saved-version URL in README and SUBMISSION.md. A local scaffold or successful CUDA test is not a completed submission.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import re

REPO_URL = "https://github.com/<platform-team>/<repository>.git"
COMMIT = ""  # Full 40-character commit from the platform-created team repository.
INCLUDE_EXPERIMENTS = False
RUN_OFFICIAL = True  # Two paired repeats, evaluation, all official checks, score display.

assert re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", REPO_URL), "Fill in a plain HTTPS team repository URL, without credentials."
assert "makazhanalpamys/soup-kernel-challenge" not in REPO_URL.lower(), "Use the platform-created team repository."
assert re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT), "Freeze and enter the complete submitted commit before running."
COMMIT = COMMIT.lower()
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
CHECKOUT = Path("/kaggle/working") / f"soup-kernel-{COMMIT[:12]}-{RUN_LABEL}"
assert not CHECKOUT.exists(), "This run directory already exists; rerun this cell to create a new run label."


In [ ]:
import os
import subprocess
import sys

listing = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True, check=True)
print(listing.stdout)
assert "T4" in listing.stdout, "Switch Kaggle's accelerator to GPU T4 x2."
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

# Exact package pins from templates/kaggle_notebook.ipynb. Do not replace
# Kaggle's PyTorch or Triton automatically; readiness records their versions.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "soup-cli[train]==0.77.0", "transformers==5.16.1", "peft==0.20.0",
    "trl==0.29.1", "bitsandbytes==0.50.2", "accelerate==1.14.0", "pytest"], check=True)


In [ ]:
import base64

token = ""
basic = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("Cloning without an optional GITHUB_TOKEN secret.")
clone_env = dict(os.environ)
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    # Transient git configuration through the child's environment: no token in
    # the clone URL, command display, repository config or credential file.
    clone_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                     GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}")
cloned = subprocess.run(["git", "clone", "-q", REPO_URL, str(CHECKOUT)],
                        env=clone_env, capture_output=True, text=True)
if cloned.returncode:
    detail = cloned.stderr
    for secret in (token, basic):
        if secret:
            detail = detail.replace(secret, "[redacted]")
    raise RuntimeError("Clone failed: " + detail)
del token, basic, clone_env
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "-q", COMMIT], check=True)
actual = subprocess.run(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert actual == COMMIT
print("Frozen checkout:", actual)
assert (CHECKOUT / "tools/run_t4_readiness.py").is_file(), "The frozen commit must include the candidate source and readiness runner."


In [ ]:
command = [sys.executable, "tools/run_t4_readiness.py",
           "--expected-commit", COMMIT, "--output-dir", "runs/readiness"]
if INCLUDE_EXPERIMENTS:
    command.append("--include-experiments")
if RUN_OFFICIAL:
    command += ["--official", "--official-out", "runs/final_candidate"]

# The runner removes TRITON_INTERPRET and fixes GPU 0 in its child processes.
# It verifies that mandatory CUDA tests passed rather than silently skipped.
# check=True is intentional: an error stops Run All before later stages.
subprocess.run(command, cwd=CHECKOUT, check=True)


In [ ]:
import json

readiness = json.loads((CHECKOUT / "runs/readiness/readiness.json").read_text())
print("Commit:", COMMIT)
print("Readiness:", readiness["status"])
print("Solution SHA256:", readiness["solution_sha256"])
print("Evidence directory:", CHECKOUT / "runs")
if RUN_OFFICIAL:
    result = json.loads((CHECKOUT / "runs/final_candidate/result.json").read_text())
    checks = json.loads((CHECKOUT / "runs/final_candidate/checks.json").read_text())
    print(json.dumps({"summary": result["summary"], "checks": checks["checks"], "notes": checks.get("notes", [])}, indent=2))
    assert len(checks["checks"]) == 10 and all(item["ok"] for item in checks["checks"])
print("Hidden-jury results and rank are not established by this public run.")


## Further experiments

The candidate exposes fixed options in `solution/settings.py`. Change them in the team repository, commit them, and freeze the new commit before collecting final evidence. This notebook never selects options based on benchmark state.

For focused screening, run `tools/benchmark_candidates.py` from a checkout. Its `--head-projections torch,t4_fused` compares the optional complete fused head path; `--suite fused --fused-rows 256,2048` measures only forward statistics. Microbenchmark ratios cannot establish the 5% useful-kernel share, trained-adapter quality, or full-command memory/time.

The official repeated order is base, solution, solution, base. It measures full commands with empty compiler caches and external sampled memory peaks. All ten checks must pass at the exact final source. Keep the full public result, checks, logs, adapters and saved notebook version; report failed gates candidly.
